# Cross-temporal LDA decoding and RSA
Stroud-style dynamic-coding test on single-object trials. Two 3-position sessions: **DMFC = Perle 2022-06-01**, **FEF = Perle 2022-06-04**. Two analyses per region: decode identity (marginal over position) and decode position (marginal over identity), 3 classes, shrinkage-LDA over a pseudopopulation (obs mask), 4-fold CV. A stable/gain code fills the off-diagonal (train-early generalises to test-late); a rotating/dynamic code concentrates accuracy near the diagonal.

In [ ]:
import sys, pathlib
root = pathlib.Path.cwd()
while not (root / 'pyproject.toml').exists():
    root = root.parent
sys.path.insert(0, str(root / 'scripts'))

import numpy as np
import matplotlib.pyplot as plt
from analysis import decode, rsa, rate_tensor, BINS

SESSIONS = {'DMFC': '2022-06-01', 'FEF': '2022-06-04'}
FACTORS = ['identity', 'position']
centers = BINS[:-1] + 0.05
ext = [0, 2, 0, 2]

In [ ]:
results = {}
for region, sess in SESSIONS.items():
    data = rate_tensor(sess, region, quality="good")          # load once per session/region
    for factor in FACTORS:
        acc, nU = decode(sess, region, factor, data=data)
        sim = rsa(sess, region, factor, data=data)
        results[(region, factor)] = dict(acc=acc, sim=sim, nU=nU)
        print(region, factor, '| units', nU, '| diag', round(np.diag(acc).mean(), 3),
              '| off-diag(train0-1s,test1-2s)', round(acc[:10, 10:].mean(), 3))

## Cross-temporal decoding matrices
Row = region, column = analysis. Dashed lines mark delay onset (1.0 s).

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(9.5, 8.5))
for r, region in enumerate(SESSIONS):
    for c, factor in enumerate(FACTORS):
        ax = axes[r, c]
        res = results[(region, factor)]
        im = ax.imshow(res['acc'], origin='lower', extent=ext, aspect='equal',
                       cmap='viridis', vmin=1/3)
        ax.axhline(1.0, color='w', lw=0.6, ls='--'); ax.axvline(1.0, color='w', lw=0.6, ls='--')
        ax.set_title(f"{region} {factor} (n={res['nU']})")
        ax.set_xlabel('test time (s)'); ax.set_ylabel('train time (s)')
        fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04, label='accuracy')
fig.suptitle('Cross-temporal LDA decoding (chance = 1/3)\nData: single-object, 3-position trials, all units, Subject P\nFEF: 2022-06-04; DMFC: 2022-06-01', fontsize=14)
fig.tight_layout()

## Within-time (diagonal) decoding

In [ ]:
fig, ax = plt.subplots(figsize=(6.5, 4))
for region in SESSIONS:
    for factor in FACTORS:
        ax.plot(centers, np.diag(results[(region, factor)]['acc']), label=f'{region} {factor}')
ax.axhline(1/3, color='k', ls=':', lw=0.8); ax.axvline(1.0, color='k', ls='--', lw=0.6)
ax.set_xlabel('time from stim onset (s)'); ax.set_ylabel('decoding accuracy')
ax.set_title('Within-time decoding'); ax.legend(fontsize=8)
fig.tight_layout()

## Cross-temporal RSA
Cross-validated cosine similarity of class-coding vectors across time. Off-diagonal decay = rotation of the coding subspace.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(9.5, 8.5))
for r, region in enumerate(SESSIONS):
    for c, factor in enumerate(FACTORS):
        ax = axes[r, c]
        sim = results[(region, factor)]['sim']
        v = float(np.nanmax(np.abs(sim)))
        im = ax.imshow(sim, origin='lower', extent=ext, aspect='equal',
                       cmap='RdBu_r', vmin=-v, vmax=v)
        ax.axhline(1.0, color='k', lw=0.5, ls='--'); ax.axvline(1.0, color='k', lw=0.5, ls='--')
        ax.set_title(f'{region} {factor} (n={res['nU']})')
        ax.set_xlabel('time (s)'); ax.set_ylabel('time (s)')
        fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04, label='cosine sim')
fig.suptitle('Cross-temporal RSA (cross-validated cosine of coding vectors)\nData: single-object, 3-position trials, all units, Subject P\nFEF: 2022-06-04; DMFC: 2022-06-01')
fig.tight_layout()